# Aligning Reuse, and Reading What Changed

**Computational Text Analysis Pack · Module 18 — Text Reuse and Intertextuality**

*Technique:* Sequence alignment on detected pairs; editing as evidence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/08-text-reuse-and-networks/18c_aligning_and_reading_reuse.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Detection tells you two texts are related. It does not tell you *how*, and for a
humanities argument the how is the whole point: what a reprinting newspaper cut,
softened or added is the evidence about its politics and its readership.

That needs **alignment** — lining the two texts up token by token so the
differences are visible. It is the same algorithm as biological sequence
alignment, which is why some text-reuse projects use BLAST outright.

## The tooling

Python's standard library ships `difflib.SequenceMatcher`, which is a perfectly
serviceable aligner for pairs. **TextPAIR** does detection and alignment in one
pipeline; **Passim** outputs aligned clusters at archive scale. For very long
sequences the biological aligners are faster, which is what Vierthaler (Chinese)
and Vesanto and Salmi (Finnish newspapers) exploited.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import difflib

def align(a, b):
    ta, tb = tokenise(a), tokenise(b)
    sm = difflib.SequenceMatcher(None, ta, tb, autojunk=False)
    ops = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        ops.append({"op": tag, "a": " ".join(ta[i1:i2]), "b": " ".join(tb[j1:j2])})
    return ops, sm.ratio()

original = ("The Governor arrived at the port on Tuesday with a large retinue "
            "and was received by the assembled chiefs, who presented gifts of "
            "ivory and gold before the customary address was read aloud.")
reprint = ("The Governor arrived at the port on Tuesday and was received by "
           "the chiefs, who presented gifts of ivory and gold. The customary "
           "address was then read aloud to the crowd.")

ops, ratio = align(original, reprint)
print(f"overall similarity: {ratio:.3f}\n")
for o in ops:
    if o["op"] == "equal":
        print(f"  =  {o['a'][:70]}")
    elif o["op"] == "delete":
        print(f"  -  {o['a'][:70]}      <- dropped in the reprint")
    elif o["op"] == "insert":
        print(f"  +  {o['b'][:70]}      <- added by the reprint")
    else:
        print(f"  ~  {o['a'][:34]!r} -> {o['b'][:34]!r}")

## Editing as a measurable object

Once alignment is mechanical, the pattern of edits becomes data. Across many
reprint pairs, what gets cut is a claim about editorial practice, and it is
countable.

In [ ]:
def shingles(text, k=4):
    toks = tokenise(text)
    return {" ".join(toks[i:i + k]) for i in range(len(toks) - k + 1)}

def jaccard(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0

docs = df.sample(n=300, random_state=8).reset_index(drop=True)
sets = [shingles(t) for t in docs["text"]]
pairs = []
for i in range(len(sets)):
    for j in range(i + 1, len(sets)):
        s = jaccard(sets[i], sets[j])
        if s < 0.95:                      # 0.95+ would be the same doc twice
            pairs.append((s, i, j))
pairs.sort(reverse=True)
# Take the most similar pairs the corpus actually has, rather than assuming a
# fixed threshold is met: how intertextual a sample is cannot be known upfront.
pairs = [p for p in pairs if p[0] > 0][:40]
print(f"{len(pairs)} pairs to align, similarity "
      f"{pairs[-1][0]:.3f} to {pairs[0][0]:.3f}\n")

dropped, added = Counter(), Counter()
lengths = []
for s, i, j in pairs:
    ops, ratio = align(docs.loc[i, "text"], docs.loc[j, "text"])
    for o in ops:
        if o["op"] == "delete":
            dropped.update(o["a"].split())
        elif o["op"] == "insert":
            added.update(o["b"].split())
    lengths.append((len(tokenise(docs.loc[i, 'text'])),
                    len(tokenise(docs.loc[j, 'text']))))

print("Words most often present in one version and absent from the other:")
for w, n in dropped.most_common(12):
    print(f"   {w:<18}{n:>4}")

In [ ]:
shorter = sum(1 for a, b in lengths if b < a)
print(f"of {len(lengths)} aligned pairs, the second version was shorter in "
      f"{shorter} ({shorter / max(len(lengths), 1):.0%})")
print()
print("Abridgement is the dominant edit in reprinting, which is what the Viral")
print("Texts work found in the American case. Whether that holds for African")
print("newspaper archives is an open question this method can answer.")

## The honest limits

Two cautions, because reuse detection produces confident-looking output on very
weak evidence.

- **Shared boilerplate is not reuse.** Datelines, agency credits and standing
  headers make unrelated articles look related; strip them before you believe a
  cluster.
- **A cluster is not a lineage.** Detection gives you an undirected graph of
  similarity. Deciding *which came first* needs dates and external evidence, and
  the usual failure is to read the highest-similarity pair as an ancestry.

In [ ]:
# Boilerplate check: which shingles are shared by an implausible number of docs?
counts = Counter()
for s in sets:
    counts.update(s)
common = [(sh, n) for sh, n in counts.most_common(15)]
print("Shingles appearing in the most documents — candidates for boilerplate:")
for sh, n in common:
    print(f"   {n:>4} docs   {sh}")
print()
print("Any of these appearing inside your 'reuse' evidence should be removed")
print("before the pair is counted, or the corpus will appear far more")
print("intertextual than it is.")

### 🔧 Your turn 1

Remove the top 20 most common shingles from every document, then re-run the pair detection. How many pairs survive?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Typically a noticeable fraction drops out, and those were the false positives —
articles sharing a wire credit or a standard sign-off rather than a story. The
survivors are the ones worth reading. This filtering step is standard in Passim
and easy to forget when rolling your own, and it is the single most common
reason a first attempt at reuse detection reports implausibly high rates.

</details>